In [ ]:
import sys
import numpy as np
import cv2
import matplotlib.pyplot as plt
import mediapipe as mp
import time
print(sys.version)
print(np.__version__)
print(mp.__version__)
print(cv2.__version__)

In [ ]:
import os

os.chdir(r"C:\Users\Elza Rose Wilson\personalworks\isl-cv-project")
print(os.getcwd())

In [11]:
Baseoptions=mp.tasks.BaseOptions
Handlandmarker=mp.tasks.vision.HandLandmarker
HandlandmarkerOptions=mp.tasks.vision.HandLandmarkerOptions
VisionRunningMode=mp.tasks.vision.RunningMode
options=HandlandmarkerOptions(
    base_options=Baseoptions(
        model_asset_path="hand_landmarker.task"
    ),
    running_mode=VisionRunningMode.VIDEO,
    num_hands=2
)
detector=Handlandmarker.create_from_options(options)


In [ ]:

BaseOptions = mp.tasks.BaseOptions
PoseLandmarker = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
RunningMode = mp.tasks.vision.RunningMode

pose_options = PoseLandmarkerOptions(
    base_options=BaseOptions(
        model_asset_path="pose_landmarker_full.task"
    ),
    running_mode=RunningMode.VIDEO,
    num_poses=1,
    min_pose_detection_confidence=0.5,
    min_pose_presence_confidence=0.5,
    min_tracking_confidence=0.5
)

pose_detector = PoseLandmarker.create_from_options(pose_options)

print("PoseLandmarker initialized successfully!")


In [13]:
# ==========================================
# FACE LANDMARKER INITIALIZATION
# ==========================================

BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
RunningMode = mp.tasks.vision.RunningMode

face_options = FaceLandmarkerOptions(
    base_options=BaseOptions(
        model_asset_path="face_landmarker.task"
    ),
    running_mode=RunningMode.VIDEO,
    num_faces=1,
    min_face_detection_confidence=0.5,
    min_face_presence_confidence=0.5,
    min_tracking_confidence=0.5,
    output_face_blendshapes=True
)

print("FaceLandmarker options configured!")

FaceLandmarker options configured!


In [14]:
import numpy as np

# 1. Pose landmarks (6)
POSE_INDICES = [11, 12, 13, 14, 23, 24]

# 2. Lip landmarks (20)
FACE_LIPS = [
    61, 146, 91, 181, 84,
    17, 314, 405, 321, 375,
    291, 409, 270, 95, 88,
    178, 87, 14, 78, 308
]

# 3. Eye landmarks (16)
FACE_EYES = [
    33, 133, 160, 158,
    153, 144, 145, 246,
    362, 263, 387, 385,
    380, 373, 374, 249
]

# 4. Eyebrow landmarks (10)
FACE_EYEBROWS = [
    107, 66, 105, 63, 70,
    336, 296, 334, 293, 300
]

# Combine facial landmark indices
FACE_INDICES = FACE_LIPS + FACE_EYES + FACE_EYEBROWS

print("Pose landmarks:", len(POSE_INDICES))
print("Lip landmarks:", len(FACE_LIPS))
print("Eye landmarks:", len(FACE_EYES))
print("Eyebrow landmarks:", len(FACE_EYEBROWS))
print("Total face landmarks:", len(FACE_INDICES))
print("Total landmarks:", 6 + 42 + len(FACE_INDICES))

Pose landmarks: 6
Lip landmarks: 20
Eye landmarks: 16
Eyebrow landmarks: 10
Total face landmarks: 46
Total landmarks: 94


In [15]:
def extract_94_landmarks(hand_result, pose_result, face_result):

    # Create an empty array for 94 landmarks.
    # Each landmark has x, y, z and visibility.
    keypoints = np.zeros((94, 4), dtype=np.float32)

    # ----------------------------------
    # 1. EXTRACT POSE LANDMARKS (0–5)
    # ----------------------------------

    if pose_result.pose_landmarks:

        pose_landmarks = pose_result.pose_landmarks[0]

        for i, idx in enumerate(POSE_INDICES):

            lm = pose_landmarks[idx]

            visibility = lm.visibility if lm.visibility is not None else 0.0

            keypoints[i] = [
                lm.x,
                lm.y,
                lm.z,
                visibility
            ]

    # ----------------------------------
    # 2. EXTRACT HAND LANDMARKS (6–47)
    # ----------------------------------

    if hand_result.hand_landmarks:

        for hand_index, hand_landmarks in enumerate(hand_result.hand_landmarks):

            if hand_index >= len(hand_result.handedness):
                continue

            classification = hand_result.handedness[hand_index][0]

            label = classification.category_name

            # Our webcam frame is currently unmirrored.
            # MediaPipe handedness assumes mirrored input.
            # Therefore, swap the labels.
            if label == "Left":
                label = "Right"

            elif label == "Right":
                label = "Left"

            # Select the correct section of our array.
            if label == "Left":
                start_idx = 6

            elif label == "Right":
                start_idx = 27

            else:
                continue

            for i, lm in enumerate(hand_landmarks):

                keypoints[start_idx + i] = [
                    lm.x,
                    lm.y,
                    lm.z,
                    1.0
                ]

    # ----------------------------------
    # 3. EXTRACT FACE LANDMARKS (48–93)
    # ----------------------------------

    if face_result.face_landmarks:

        face_landmarks = face_result.face_landmarks[0]

        for i, idx in enumerate(FACE_INDICES):

            lm = face_landmarks[idx]

            visibility = 1.0 if lm.z > -0.5 else 0.5

            keypoints[48 + i] = [
                lm.x,
                lm.y,
                lm.z,
                visibility
            ]

    return keypoints

In [16]:

import numpy as np

# Finger points used by the hand-feature calculations
finger_points = {
    "Thumb": (1, 2, 4),
    "Index": (5, 6, 8),
    "Middle": (9, 10, 12),
    "Ring": (13, 14, 16),
    "Little": (17, 18, 20)
}

# Pairs used for finger-distance calculations
finger_pairs = {
    "Thumb-Index": (4, 8),
    "Index-Middle": (8, 12),
    "Middle-Ring": (12, 16),
    "Ring-Little": (16, 20)
}


def calculate_distance(point1, point2):
    return np.linalg.norm(
        np.array(point1, dtype=np.float32)
        - np.array(point2, dtype=np.float32)
    )


def normalize_distance(distance, hand_size):
    if hand_size < 1e-8:
        return 0.0

    return distance / hand_size


def calculate_angle(a, b, c):
    a = np.array(a, dtype=np.float32)
    b = np.array(b, dtype=np.float32)
    c = np.array(c, dtype=np.float32)

    ba = a - b
    bc = c - b

    denominator = (
        np.linalg.norm(ba) * np.linalg.norm(bc)
    )

    if denominator < 1e-8:
        return 0.0

    cosine = np.dot(ba, bc) / denominator
    cosine = np.clip(cosine, -1.0, 1.0)

    return np.degrees(np.arccos(cosine))


def extract_features(hand_landmarks):

    wrist = [
        hand_landmarks[0].x,
        hand_landmarks[0].y
    ]

    middle_base = [
        hand_landmarks[9].x,
        hand_landmarks[9].y
    ]

    hand_size = calculate_distance(wrist, middle_base)

    features = []

    # Five finger angles
    for finger, points in finger_points.items():

        a_idx, b_idx, c_idx = points

        a = [
            hand_landmarks[a_idx].x,
            hand_landmarks[a_idx].y
        ]

        b = [
            hand_landmarks[b_idx].x,
            hand_landmarks[b_idx].y
        ]

        c = [
            hand_landmarks[c_idx].x,
            hand_landmarks[c_idx].y
        ]

        features.append(calculate_angle(a, b, c))

    # Four normalized finger distances
    for pair_name, points in finger_pairs.items():

        p1, p2 = points

        a = [
            hand_landmarks[p1].x,
            hand_landmarks[p1].y
        ]

        b = [
            hand_landmarks[p2].x,
            hand_landmarks[p2].y
        ]

        distance = calculate_distance(a, b)

        features.append(
            normalize_distance(distance, hand_size)
        )

    return features


# Verify all helper functions
print("Distance test:", calculate_distance([0, 0], [3, 4]))
print("Angle test:", calculate_angle([0, 1], [0, 0], [1, 0]))
print("Feature count:", len(finger_points) + len(finger_pairs))
print("All helper functions loaded!")


Distance test: 5.0
Angle test: 90.0
Feature count: 9
All helper functions loaded!


In [ ]:

# ============================================================
# HAND + POSE + FACE DETECTION + 94 LANDMARK RECORDING
# ============================================================

import cv2
import numpy as np
import time


# ============================================================
# 1. CAMERA
# ============================================================

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    raise RuntimeError("Unable to open webcam.")

start_time = time.monotonic()
frame_count = 0
previous_time = time.time()


# ============================================================
# 2. INITIALIZE DETECTORS
# ============================================================

try:
    detector.close()
except (NameError, ValueError):
    pass

detector = Handlandmarker.create_from_options(options)


try:
    pose_detector.close()
except (NameError, ValueError):
    pass

pose_detector = PoseLandmarker.create_from_options(pose_options)


try:
    face_detector.close()
except (NameError, ValueError):
    pass

face_detector = FaceLandmarker.create_from_options(face_options)


# ============================================================
# 3. HAND CONNECTIONS
# ============================================================

connections = [
    (0, 1), (1, 2), (2, 3), (3, 4),
    (0, 5), (5, 6), (6, 7), (7, 8),
    (0, 9), (9, 10), (10, 11), (11, 12),
    (0, 13), (13, 14), (14, 15), (15, 16),
    (0, 17), (17, 18), (18, 19), (19, 20),
    (5, 9), (9, 13), (13, 17)
]


# ============================================================
# 4. POSE CONNECTIONS
# ============================================================

POSE_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 7),
    (0, 4), (4, 5), (5, 6), (6, 8),
    (9, 10),
    (11, 12),
    (11, 13), (13, 15),
    (15, 17), (15, 19), (15, 21),
    (17, 19),
    (12, 14), (14, 16),
    (16, 18), (16, 20), (16, 22),
    (18, 20),
    (11, 23), (12, 24),
    (23, 24),
    (23, 25), (25, 27), (27, 29), (29, 31),
    (27, 31),
    (24, 26), (26, 28), (28, 30), (30, 32),
    (28, 32)
]


# ============================================================
# 5. HAND FEATURE NAMES
# ============================================================

feature_names = [
    "Thumb_Angle",
    "Index_Angle",
    "Middle_Angle",
    "Ring_Angle",
    "Little_Angle",
    "Thumb_Index_Distance",
    "Index_Middle_Distance",
    "Middle_Ring_Distance",
    "Ring_Little_Distance"
]

latest_features = None
latest_blendshapes = None


# ============================================================
# 6. RECORDING VARIABLES
# ============================================================

# Stores one (94, 4) array for every recorded frame.
keypoint_sequence = []

# Recording state
recording = False


# ============================================================
# 7. SAVE FUNCTION
# ============================================================

def save_keypoint_sequence(sequence):

    if len(sequence) == 0:
        print("No frames were recorded.")
        return

    sequence_array = np.array(
        sequence,
        dtype=np.float32
    )

    print("\n========== SEQUENCE DETAILS ==========")

    print("Sequence shape:", sequence_array.shape)
    print("Data type:", sequence_array.dtype)
    print("Total frames:", sequence_array.shape[0])
    print("Landmarks per frame:", sequence_array.shape[1])
    print("Values per landmark:", sequence_array.shape[2])

    np.save("islm_sequence.npy", sequence_array)

    print("\nSaved successfully as islm_sequence.npy")


# ============================================================
# 8. MAIN WEBCAM LOOP
# ============================================================

try:

    while True:

        # ----------------------------------------------------
        # READ FRAME
        # ----------------------------------------------------

        ret, frame = cap.read()

        if not ret:
            print("Unable to read webcam frame.")
            break

        height, width, _ = frame.shape

        frame_rgb = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )

        mp_img = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=frame_rgb
        )

        timestamp_ms = frame_count * 33
        frame_count += 1


        # ====================================================
        # 9. HAND DETECTION
        # ====================================================

        result = detector.detect_for_video(
            mp_img,
            timestamp_ms
        )


        # ====================================================
        # 10. POSE DETECTION
        # ====================================================

        pose_result = pose_detector.detect_for_video(
            mp_img,
            timestamp_ms
        )

        if pose_result.pose_landmarks:

            pose_landmarks = pose_result.pose_landmarks[0]

            # Draw pose connections

            for start_idx, end_idx in POSE_CONNECTIONS:

                start_point = pose_landmarks[start_idx]
                end_point = pose_landmarks[end_idx]

                x1 = int(start_point.x * width)
                y1 = int(start_point.y * height)

                x2 = int(end_point.x * width)
                y2 = int(end_point.y * height)

                if (
                    0 <= x1 < width and
                    0 <= y1 < height and
                    0 <= x2 < width and
                    0 <= y2 < height
                ):

                    cv2.line(
                        frame,
                        (x1, y1),
                        (x2, y2),
                        (0, 255, 0),
                        2
                    )

            # Draw pose landmarks

            for i, landmark in enumerate(pose_landmarks):

                x = int(landmark.x * width)
                y = int(landmark.y * height)

                if 0 <= x < width and 0 <= y < height:

                    cv2.circle(
                        frame,
                        (x, y),
                        4,
                        (0, 255, 0),
                        -1
                    )

            cv2.putText(
                frame,
                f"Pose: {len(pose_landmarks)}",
                (10, height - 20),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 255, 0),
                2
            )


        # ====================================================
        # 11. FACE DETECTION
        # ====================================================

        face_result = face_detector.detect_for_video(
            mp_img,
            timestamp_ms
        )

        latest_blendshapes = None

        if face_result.face_landmarks:

            face_landmarks = face_result.face_landmarks[0]

            # Draw face landmarks

            for landmark in face_landmarks:

                x = int(landmark.x * width)
                y = int(landmark.y * height)

                if 0 <= x < width and 0 <= y < height:

                    cv2.circle(
                        frame,
                        (x, y),
                        1,
                        (255, 255, 0),
                        -1
                    )

            cv2.putText(
                frame,
                f"Face: {len(face_landmarks)}",
                (10, height - 45),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (255, 255, 0),
                2
            )

            # Read expression blendshapes

            if face_result.face_blendshapes:

                latest_blendshapes = face_result.face_blendshapes[0]

                cv2.putText(
                    frame,
                    f"Blendshapes: {len(latest_blendshapes)}",
                    (10, height - 70),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    (255, 255, 0),
                    2
                )


        # ====================================================
        # 12. EXTRACT 94 LANDMARKS
        # ====================================================

        frame_keypoints = extract_94_landmarks(
            result,
            pose_result,
            face_result
        )


        # ====================================================
        # 13. COLLECT LANDMARKS WHILE RECORDING
        # ====================================================

        if recording:

            keypoint_sequence.append(
                frame_keypoints.copy()
            )


        # ====================================================
        # 14. HAND FEATURE PROCESSING
        # ====================================================

        latest_features = None

        processing_time = time.time() - previous_time
        previous_time = time.time()

        for hand_index, handlandmarks in enumerate(result.hand_landmarks):

            # Draw hand landmarks

            for i, landmark in enumerate(handlandmarks):

                x = int(landmark.x * width)
                y = int(landmark.y * height)

                cv2.circle(
                    frame,
                    (x, y),
                    5,
                    (0, 0, 255),
                    -1
                )

                cv2.putText(
                    frame,
                    str(i),
                    (x + 5, y - 5),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.4,
                    (255, 0, 0),
                    1
                )

            wrist = [
                handlandmarks[0].x,
                handlandmarks[0].y
            ]

            middle_base = [
                handlandmarks[9].x,
                handlandmarks[9].y
            ]

            hand_size = calculate_distance(
                wrist,
                middle_base
            )

            # Draw hand connections

            for start, end in connections:

                x1 = int(handlandmarks[start].x * width)
                y1 = int(handlandmarks[start].y * height)

                x2 = int(handlandmarks[end].x * width)
                y2 = int(handlandmarks[end].y * height)

                cv2.line(
                    frame,
                    (x1, y1),
                    (x2, y2),
                    (0, 0, 255),
                    2
                )

            # Finger angles and status

            for finger_index, (finger, points) in enumerate(
                finger_points.items()
            ):

                a_idx, b_idx, c_idx = points

                a = [
                    handlandmarks[a_idx].x,
                    handlandmarks[a_idx].y
                ]

                b = [
                    handlandmarks[b_idx].x,
                    handlandmarks[b_idx].y
                ]

                c = [
                    handlandmarks[c_idx].x,
                    handlandmarks[c_idx].y
                ]

                angle = calculate_angle(a, b, c)

                status = "Extended" if angle > 160 else "Bent"

                x_position = 10 if hand_index == 0 else width - 220
                y_position = 30 + finger_index * 30

                cv2.putText(
                    frame,
                    f"{finger}: {status}",
                    (x_position, y_position),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    (255, 0, 0),
                    2
                )

            # Finger distances

            for pair_index, (pair_name, points) in enumerate(
                finger_pairs.items()
            ):

                point1, point2 = points

                a = [
                    handlandmarks[point1].x,
                    handlandmarks[point1].y
                ]

                b = [
                    handlandmarks[point2].x,
                    handlandmarks[point2].y
                ]

                distance = calculate_distance(a, b)

                normalized_distance = normalize_distance(
                    distance,
                    hand_size
                )

                x_position = 10 if hand_index == 0 else width - 220
                y_position = 210 + pair_index * 25

                cv2.putText(
                    frame,
                    f"{pair_name}: {normalized_distance:.3f}",
                    (x_position, y_position),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.4,
                    (255, 0, 0),
                    1
                )

            # Extract nine hand features

            features = extract_features(handlandmarks)

            latest_features = features


        # ====================================================
        # 15. DISPLAY RECORDING STATUS
        # ====================================================

        if recording:

            cv2.putText(
                frame,
                f"RECORDING | Frames: {len(keypoint_sequence)}",
                (10, 100),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (0, 0, 255),
                2
            )

        else:

            cv2.putText(
                frame,
                "NOT RECORDING",
                (10, 100),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (255, 255, 255),
                2
            )


        # ====================================================
        # 16. DISPLAY WEBCAM
        # ====================================================

        cv2.imshow("MediaPipe Detection", frame)

        key = cv2.waitKey(10) & 0xFF


        # ====================================================
        # 17. KEYBOARD CONTROLS
        # ====================================================

        # Press R to start or stop recording

        if key == ord('r'):

            if not recording:

                # START RECORDING

                keypoint_sequence = []
                recording = True

                print("\nRecording started...")

            else:

                # STOP RECORDING

                recording = False

                print("\nRecording stopped.")

                save_keypoint_sequence(
                    keypoint_sequence
                )


        # Press S to inspect the current frame

        elif key == ord('s'):

            print("\n========== CURRENT FRAME ==========")

            print("Shape:", frame_keypoints.shape)
            print("Data type:", frame_keypoints.dtype)

            print(
                "Pose:",
                np.sum(frame_keypoints[0:6, 3] > 0.5)
            )

            print(
                "Left hand:",
                np.sum(frame_keypoints[6:27, 3] > 0.5)
            )

            print(
                "Right hand:",
                np.sum(frame_keypoints[27:48, 3] > 0.5)
            )

            print(
                "Face:",
                np.sum(frame_keypoints[48:94, 3] > 0.5)
            )


        # Press Q to exit

        elif key == ord('q'):

            # Save unfinished recording before exiting

            if recording:

                recording = False

                print("\nSaving recording before exit...")

                save_keypoint_sequence(
                    keypoint_sequence
                )

            break


# ============================================================
# 18. CLEANUP
# ============================================================

finally:

    cap.release()

    cv2.destroyAllWindows()

    detector.close()
    pose_detector.close()
    face_detector.close()

    print("\nAll three detectors closed.")


In [ ]:
import numpy as np

sequence = np.load("islm_sequence.npy")

print("Shape:", sequence.shape)
print("Dtype:", sequence.dtype)

In [ ]:
import numpy as np

sequence = np.load("islm_sequence.npy")

print("Shape:", sequence.shape)
print("Data type:", sequence.dtype)
print("NaN:", np.isnan(sequence).any())
print("Min:", sequence.min())
print("Max:", sequence.max())

In [ ]:
import numpy as np

sequence = np.load("islm_sequence.npy")

# Landmark visibility
visibility = sequence[:, :, 3]

# Official ISLiM landmark indices
left_hand = sequence[:, 6:27, :]
right_hand = sequence[:, 27:48, :]

print("Total frames:", len(sequence))

print("\nLEFT HAND")
print("Average visible landmarks:",
      np.mean(left_hand[:, :, 3] > 0.5) * 21)

print("\nRIGHT HAND")
print("Average visible landmarks:",
      np.mean(right_hand[:, :, 3] > 0.5) * 21)

print("\nOverall visibility:")
print("Pose:", np.mean(visibility[:, :6] > 0.5))
print("Left hand:", np.mean(visibility[:, 6:27] > 0.5))
print("Right hand:", np.mean(visibility[:, 27:48] > 0.5))

In [30]:

def interpolate_keypoints(keypoint_sequence):

    keypoints = np.asarray(
        keypoint_sequence,
        dtype=np.float32
    ).copy()

    T, K, C = keypoints.shape

    for k in range(K):

        valid_frames = np.where(
            keypoints[:, k, 3] > 0.5
        )[0]

        # No valid detection for this landmark
        if len(valid_frames) == 0:
            keypoints[:, k, :] = 0.0
            continue

        # 1. Interpolate short internal gaps
        for i in range(len(valid_frames) - 1):

            start = valid_frames[i]
            end = valid_frames[i + 1]

            gap = end - start

            if 1 < gap <= 5:

                for j in range(1, gap):

                    alpha = j / gap

                    keypoints[start + j, k, :3] = (
                        (1 - alpha) * keypoints[start, k, :3]
                        + alpha * keypoints[end, k, :3]
                    )

                    keypoints[start + j, k, 3] = (
                        (1 - alpha) * keypoints[start, k, 3]
                        + alpha * keypoints[end, k, 3]
                    )

        # 2. Fill beginning frames
        first_valid = valid_frames[0]

        for f in range(first_valid):
            if keypoints[f, k, 3] < 0.5:
                keypoints[f, k] = keypoints[first_valid, k]

        # 3. Fill ending frames
        last_valid = valid_frames[-1]

        for f in range(last_valid + 1, T):
            if keypoints[f, k, 3] < 0.5:
                keypoints[f, k] = keypoints[last_valid, k]

    # 4. Set remaining invalid landmarks to zero
    invalid_mask = keypoints[:, :, 3] < 0.5
    keypoints[invalid_mask] = 0.0

    return keypoints


print("Corrected interpolation function loaded!")


Corrected interpolation function loaded!


In [31]:
# Apply official ISLiM interpolation
interpolated_sequence = interpolate_keypoints(sequence)

print("Original shape:", sequence.shape)
print("Interpolated shape:", interpolated_sequence.shape)

print("Original dtype:", sequence.dtype)
print("Interpolated dtype:", interpolated_sequence.dtype)

Original shape: (273, 94, 4)
Interpolated shape: (273, 94, 4)
Original dtype: float32
Interpolated dtype: float32


In [32]:
# Count invalid landmark entries using visibility
original_missing = np.sum(sequence[:, :, 3] <= 0.5)

processed_missing = np.sum(
    interpolated_sequence[:, :, 3] <= 0.5
)

print("Missing landmarks before:", original_missing)
print("Missing landmarks after:", processed_missing)
print(
    "Landmarks recovered:",
    original_missing - processed_missing
)

Missing landmarks before: 7155
Missing landmarks after: 4355
Landmarks recovered: 2800


In [33]:
np.save("islm_sequence_raw.npy", sequence)

np.save(
    "islm_sequence_interpolated.npy",
    interpolated_sequence
)

print("Raw and interpolated sequences saved.")

Raw and interpolated sequences saved.


In [34]:

import numpy as np
import torch

MAX_FRAMES = 128

def prepare_islim_input(keypoint_sequence):

    # Work on a copy
    keypoints = np.asarray(
        keypoint_sequence,
        dtype=np.float32
    ).copy()

    # Expected input: (T, 94, 4)
    T, K, D = keypoints.shape

    assert K == 94, f"Expected 94 landmarks, got {K}"
    assert D == 4, f"Expected 4 values per landmark, got {D}"

    # -----------------------------------------
    # 1. NORMALIZE X, Y AND Z
    # -----------------------------------------

    for coord in range(3):

        arr = keypoints[:, :, coord]

        minimum = arr.min()
        maximum = arr.max()

        value_range = maximum - minimum

        if value_range > 1e-8:
            keypoints[:, :, coord] = (
                arr - minimum
            ) / value_range
        else:
            keypoints[:, :, coord] = arr - minimum

    # -----------------------------------------
    # 2. TEMPORAL SAMPLING / PADDING
    # -----------------------------------------

    original_frames = T

    if T > MAX_FRAMES:

        indices = torch.linspace(
            0,
            T - 1,
            MAX_FRAMES
        ).long().numpy()

        keypoints = keypoints[indices]

    elif T < MAX_FRAMES:

        padding = np.zeros(
            (MAX_FRAMES - T, 94, 4),
            dtype=np.float32
        )

        keypoints = np.concatenate(
            [keypoints, padding],
            axis=0
        )

    # -----------------------------------------
    # 3. CHANGE DIMENSION ORDER
    # -----------------------------------------

    # (128, 94, 4) -> (4, 94, 128)
    keypoints = np.transpose(
        keypoints,
        (2, 1, 0)
    ).copy()

    # Add batch dimension
    # (4, 94, 128) -> (1, 4, 94, 128)

    keypoints = np.expand_dims(keypoints, axis=0)

    return torch.from_numpy(keypoints).float(), min(
        original_frames,
        MAX_FRAMES
    )


# Prepare your interpolated recording
model_input, valid_frames = prepare_islim_input(
    interpolated_sequence
)

print("Model input shape:", model_input.shape)
print("Model input dtype:", model_input.dtype)
print("Valid frames:", valid_frames)


Model input shape: torch.Size([1, 4, 94, 128])
Model input dtype: torch.float32
Valid frames: 128


In [35]:
print("Shape:", tuple(model_input.shape))
print("Dtype:", model_input.dtype)
print("Contains NaN:", torch.isnan(model_input).any().item())
print("Contains Inf:", torch.isinf(model_input).any().item())

print(
    "X range:",
    model_input[0, 0].min().item(),
    model_input[0, 0].max().item()
)

print(
    "Y range:",
    model_input[0, 1].min().item(),
    model_input[0, 1].max().item()
)

print(
    "Z range:",
    model_input[0, 2].min().item(),
    model_input[0, 2].max().item()
)

Shape: (1, 4, 94, 128)
Dtype: torch.float32
Contains NaN: False
Contains Inf: False
X range: 0.0 1.0
Y range: 0.0 1.0
Z range: 0.17562167346477509 0.9700404405593872


In [36]:
np.save(
    "islm_model_input.npy",
    model_input.cpu().numpy()
)

print("Saved: islm_model_input.npy")

Saved: islm_model_input.npy


In [35]:
from IPython.display import FileLink, display

display(FileLink("islm_model_input.npy"))

C:\Users\Elza Rose Wilson\personalworks\isl-cv-project\islm_model_input.npy

In [40]:
print("Local input:")
print("Mean:", model_input.mean().item())
print("Std:", model_input.std().item())

Local input:
Mean: 0.6530594229698181
Std: 0.3305054306983948
